In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
import h5py

from scipy.interpolate import CubicSpline

from tqdm.auto import tqdm

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

from multiprocess import Pool

In [ ]:
import pyanalib.pandas_helpers as ph
from makedf.util import *

import kinematics
import gump_cuts as gc
import loaddf
import syst

import importlib

In [ ]:
importlib.reload(gc)

In [ ]:
PLOTDIR = "/Users/gputnam/Work/osc/cafpyana/signal-box-syst-plots-5-5/"

DOSAVE = True

import os
os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(PLOTDIR + "/png", exist_ok=True)
os.makedirs(PLOTDIR + "/pdf", exist_ok=True)

In [ ]:
FONTSIZE = 14
HAWKS_COLORS = ["#315031", "#d54c28", "#1e3f54", "#c89648", "#43140b", "#95af8b"]

def add_style(ax, xlabel, title="", det="ICARUS", ylabel='Events / $10^{20}$ POT', legend_loc=None, legend_ncol=1, legend_title=None):
    ax.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_title(f"$\\bf{{{det}}}$  {title}", fontsize=FONTSIZE+2)
    ax.legend(fontsize=FONTSIZE-1, loc=legend_loc, ncol=legend_ncol, title=legend_title, title_fontsize=FONTSIZE)

In [ ]:
RECO = "PANDORA"

In [ ]:
DF_DIR = "/Volumes/My Passport/gump/sbn-rewgted-7/"

SCV_FILES = [DF_DIR + "SBND_SpringMC_rewgt_E_%i.df" % i for i in range(20)]
SDIRT_FILE = DF_DIR + "SBND_SpringLowEMC.df"
SBEAMOFF_FILE = DF_DIR + "SBND_SpringBNBOffData_5000.df"
SDETVAR_FILES = [
    DF_DIR + "SBND_SpringMC_Nom.df",
    DF_DIR + "SBND_SpringMC_WMXThetaXW.df",
    DF_DIR + "SBND_SpringMC_WMYZ.df",
    # DF_DIR + "SBND_SpringMC_0xSCE.df",
    DF_DIR + "SBND_SpringMC_2xSCE.df",
]
SDETVAR_NAMES = ["Nominal",
                 "WM $X\\theta_{xw}$", "WM $YZ$", 
                 # "0x SCE", 
                 "2x SCE"]


IRUN2_CV_FILES = [DF_DIR + "ICARUS_SpringMCOverlay_rewgt.df"]
IRUN4_CV_FILES = [DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(10)]
IRUN2_DIRT_FILES = [DF_DIR + "ICARUS_Spring_Overlay_Dirt_lowE.df"]
IRUN4_DIRT_FILES = [DF_DIR + "ICARUSRun4_Spring_Overlay_Dirt_lowE.df"]
IRUN2_BEAMOFF_FILE = DF_DIR + "ICARUS_SpringRun2BNBOff_unblind_prescaled.df"
IRUN4_BEAMOFF_FILE = DF_DIR + "ICARUS_SpringRun4BNBOff_unblind_prescaled.df"

IRUN2_DETVAR_FILES = [
                DF_DIR + "ICARUS_SpringMCOverlay.df", 
                # DF_DIR + "ICARUS_Spring_WMXTHXW_BAD.df", 
                DF_DIR + "ICARUS_Spring_WMXTHXW.df", 
                DF_DIR + "ICARUS_Spring_SCE.df",
    # HANNAH-- don't actually apply the 2x SCE -- this isn't there as a one-sigma variation
                DF_DIR + "ICARUS_Spring_SCE2x.df",
    ]

# HANNAH-- note here-- I'm just using the same sample over-and-over for the ICARUSRun4 "variations"
# you'll need to replace this with some logic to scale the Run2 variations to the full POT
IRUN4_DETVAR_FILES = [
                DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_0.df",
                DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_0.df",
                DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_0.df",
                DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_0.df",
]

IDETVAR_NAMES = ["Nominal",
                 "WM $X\\theta_{xw}$",
                 "SCE",
                 "SCE 2x",
                ]

IRUN2_POT = 2e20
IRUN4_POT = 3e20

In [ ]:
# IDdf, IDmatch, IDpot = loaddf.load(DF_DIR + "ICARUS_Spring_WMXTHXW_rewgt.df", include_syst=False)

In [ ]:
IGOAL_POT = IRUN2_POT + IRUN4_POT
SGOAL_POT = 1e20

In [ ]:
def FV(df, det):    
    is_spine = "SPINE" in RECO
    
    ret = gc.slcfv_cut(df, det) & gc.mufv_cut(df, det) & gc.pfv_cut(df, det) 
    
    if is_spine:
        ret = ret & (df.is_time_contained)
    
    return ret

def FVSBND(df):
    return FV(df, "SBND") & gc.flash_cut(df, "SBND")

def FVICARUSRun2(df):
    return FV(df, "ICARUS Run2") & gc.flash_cut(df, "ICARUS Run2")

def FVICARUSRun4(df):
    return FV(df, "ICARUS Run4") & gc.flash_cut(df, "ICARUS Run4")

In [ ]:
def TrueAV(df, det):
    vtx = pd.DataFrame({'x': df.true_vtx_x,
                       'y': df.true_vtx_y,
                       'z': df.true_vtx_z}, index=df.index)
    return gc._fv_cut(vtx, det, 0, 0, 0, 0)

def TrueAVSBND(df):
    return TrueAV(df, "SBND")

def TrueAVICARUSRun2(df):
    return TrueAV(df, "ICARUS Run2")

def TrueAVICARUSRun4(df):
    return TrueAV(df, "ICARUS Run4")

def OOAVSBND(df):
    return ~np.isnan(df.true_vtx_x) & ~TrueAVSBND(df)

def OOAVICARUS(df):
    return ~np.isnan(df.true_vtx_x) & ~TrueAVICARUSRun4(df)

In [ ]:
importlib.reload(loaddf)
importlib.reload(gc)
importlib.reload(syst)

In [ ]:
# Load Run 2 and Run 4 ICARUS MC separately
Idf_r2, Imatch_r2, Ipot_r2 = loaddf.loadl(IRUN2_CV_FILES, njob=min(len(IRUN2_CV_FILES), 10), detector="ICARUS Run2", 
                                 preselection=FVICARUSRun2, reweight_aFF=True)
Idf_r4, Imatch_r4, Ipot_r4 = loaddf.loadl(IRUN4_CV_FILES, njob=min(len(IRUN4_CV_FILES), 10), detector="ICARUS Run4", 
                                 preselection=FVICARUSRun4, reweight_aFF=True)

# Scale each run to its target POT before combining
loaddf.scale_pot(Idf_r2, Ipot_r2, IRUN2_POT)
loaddf.scale_pot(Idf_r4, Ipot_r4, IRUN4_POT)
Idf = pd.concat([Idf_r2, Idf_r4]).reset_index(drop=True)
Imatch = pd.concat([Imatch_r2, Imatch_r4])
Ipot = IRUN2_POT + IRUN4_POT

# Load Run 2 and Run 4 ICARUS dirt separately
if len(IRUN2_DIRT_FILES) > 0 and len(IRUN4_DIRT_FILES) > 0:
    Idirt_r2, Idirtmatch_r2, Idirtpot_r2 = loaddf.loadl(IRUN2_DIRT_FILES, njob=min(len(IRUN2_DIRT_FILES), 10), detector="ICARUS Run2", 
                                               preselection=FVICARUSRun2, include_syst=False)
    Idirt_r4, Idirtmatch_r4, Idirtpot_r4 = loaddf.loadl(IRUN4_DIRT_FILES, njob=min(len(IRUN4_DIRT_FILES), 10), detector="ICARUS Run4", 
                                               preselection=FVICARUSRun4, include_syst=False)
    loaddf.scale_pot(Idirt_r2, Idirtpot_r2, IRUN2_POT)
    loaddf.scale_pot(Idirt_r4, Idirtpot_r4, IRUN4_POT)
    Idirt = pd.concat([Idirt_r2, Idirt_r4]).reset_index(drop=True)
else:
    Idirt = pd.DataFrame(columns=Idf.columns)
Idirtpot = IRUN2_POT + IRUN4_POT

# Load Run 2 and Run 4 ICARUS beamoff separately
Ioffbeam_r2, _, Ioffbeampot_r2 = loaddf.load(IRUN2_BEAMOFF_FILE, detector="ICARUS Run2", offbeampot=True, 
                                                   preselection=FVICARUSRun2, include_syst=False, load_truth=False)
Ioffbeam_r4, _, Ioffbeampot_r4 = loaddf.load(IRUN4_BEAMOFF_FILE, detector="ICARUS Run4", offbeampot=True, 
                                                   preselection=FVICARUSRun4, include_syst=False, load_truth=False)
loaddf.scale_pot(Ioffbeam_r2, Ioffbeampot_r2, IRUN2_POT)
loaddf.scale_pot(Ioffbeam_r4, Ioffbeampot_r4, IRUN4_POT)
Ioffbeam = pd.concat([Ioffbeam_r2, Ioffbeam_r4]).reset_index(drop=True)
Ioffbeampot = IRUN2_POT + IRUN4_POT

In [ ]:
Sdf, Smatch, Spot = loaddf.loadl(SCV_FILES, njob=min(len(SCV_FILES), 10), detector="SBND",
                                 preselection=FVSBND, reweight_aFF=True)
Sdirt, Sdirtmatch, Sdirtpot = loaddf.load(SDIRT_FILE, detector="SBND",
                                          preselection=FVSBND, include_syst=False)
Soffbeam, Soffbeammatch, Soffbeampot = loaddf.load(SBEAMOFF_FILE, detector="SBND",
                                                   preselection=FVSBND, offbeampot=True, include_syst=False, load_truth=False)

In [ ]:
Sdetvars, Sdetvarsmatch, Sdetvar_pots = zip(*tqdm([loaddf.load(f, preselection=FVSBND, include_syst=False, detector="SBND") for f in SDETVAR_FILES]))

Sdetvars, Sdetvar_pots = loaddf.match_common_evts(Sdetvarsmatch, Sdetvars, Sdetvar_pots)

Ir2detvars, Ir2detvarsmatch, Ir2detvar_pots = zip(*tqdm([loaddf.load(f, preselection=FVICARUSRun2, include_syst=False, detector="ICARUS Run2") for f in IRUN2_DETVAR_FILES]))
Ir4detvars, Ir4detvarsmatch, Ir4detvar_pots = zip(*tqdm([loaddf.load(f, preselection=FVICARUSRun4, include_syst=False, detector="ICARUS Run4") for f in IRUN4_DETVAR_FILES]))

Ir2detvars, Ir2detvar_pots = loaddf.match_common_evts(Ir2detvarsmatch, Ir2detvars, Ir2detvar_pots)
Ir4detvars, Ir4detvar_pots = loaddf.match_common_evts(Ir4detvarsmatch, Ir4detvars, Ir4detvar_pots)



In [ ]:
for c in Sdf.columns:
    if "_univ" in c:
        Sdirt[c] = 1

for c in Idf.columns:
    if "_univ" in c:
        Idirt[c] = 1

if "dirt" not in Sdf.columns:
    Sdf["dirt"] = False
    Sdirt["dirt"] = True

if "dirt" not in Idf.columns:
    Idf["dirt"] = False
    Idirt["dirt"] = True

In [ ]:
for i in range(len(Ir2detvars)):
    loaddf.scale_pot(Ir2detvars[i], Ir2detvar_pots[i], IRUN2_POT)

for i in range(len(Ir4detvars)):
    loaddf.scale_pot(Ir4detvars[i], Ir4detvar_pots[i], IRUN2_POT)

In [ ]:
Idetvars = []

for i in range(len(Ir4detvars)):
    Idetvars.append(pd.concat([Ir2detvars[i], Ir4detvars[i]]).reset_index(drop=True))

In [ ]:
print("SBND CV")
loaddf.scale_pot(Sdf, Spot, SGOAL_POT)
print("SBND Dirt")
loaddf.scale_pot(Sdirt, Sdirtpot, SGOAL_POT)
print("SBND Beam OFF")
loaddf.scale_pot(Soffbeam, Soffbeampot, SGOAL_POT)
for i in range(len(Sdetvars)):
    print("SBND", SDETVAR_NAMES[i])
    loaddf.scale_pot(Sdetvars[i], Sdetvar_pots[i], SGOAL_POT)

In [ ]:
print("ICARUS CV -- already scaled during load")
print("ICARUS Dirt -- already scaled during load")
print("ICARUS Detvars -- already scaled during load")
print("ICARUS Beam OFF -- already scaled during load")

In [ ]:
Sdf = pd.concat([Sdf[~Sdf.dirt], Sdirt])
Idf = pd.concat([Idf[~Idf.dirt], Idirt])

In [ ]:
chi2vars = [
    "mu_chi2_of_mu_cand",
    "prot_chi2_of_mu_cand",
    "mu_chi2_of_prot_cand",
    "prot_chi2_of_prot_cand",
]

chi2labels = [
    "$\\chi^2_\\mu$ of Muon Candidate",
    "$\\chi^2_p$ of Muon Candidate",
    "$\\chi^2_\\mu$ of Proton Candidate",
    "$\\chi^2_p$ of Proton Candidate",
]

chi2bins = [
    np.linspace(0, 60, 21),
    np.linspace(0, 300, 21),
    np.linspace(0, 60, 21),
    np.linspace(0, 300, 21)
]

In [ ]:
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    plt.figure(i)
    _ = plt.hist(Sdf[c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
    _ = plt.hist(Sdf[c.replace("chi2", "chi2smear13")], bins=b, density=True, histtype="step", linewidth=2, label="Smeared dE/dx")
    _ = plt.hist(Sdf[c.replace("chi2", "chi2hi")], bins=b, density=True, histtype="step", linewidth=2, label="Gain Hi")
    # _ = plt.hist(Sdf[c.replace("chi2", "chi22hi")], bins=b, density=True, histtype="step", linewidth=2, label="Gain Hi (2x)")
    add_style(plt.gca(), l, det="SBND", ylabel="Area Normalized")

    if DOSAVE:
        plt.savefig(PLOTDIR + "/png/SBND_%s_variations.png" % c, bbox_inches="tight")
        plt.savefig(PLOTDIR + "/pdf/SBND_%s_variations.pdf" % c, bbox_inches="tight")

In [ ]:

for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    plt.figure(i)
    _ = plt.hist(Idf[c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
    _ = plt.hist(Idf[c.replace("chi2", "chi2smear13")], bins=b, density=True, histtype="step", linewidth=2, label="Smeared dE/dx")
    _ = plt.hist(Idf[c.replace("chi2", "chi2hi")], bins=b, density=True, histtype="step", linewidth=2, label="Gain Hi")
    add_style(plt.gca(), l, det="ICARUS", ylabel="Area Normalized")

    if DOSAVE:
        plt.savefig(PLOTDIR + "/png/ICARUS_%s_variations.png" % c, bbox_inches="tight")
        plt.savefig(PLOTDIR + "/pdf/ICARUS_%s_variations.pdf" % c, bbox_inches="tight")

In [ ]:
for i, (c,l,b) in enumerate(zip(chi2vars, chi2labels, chi2bins)):
    plt.figure(i)
    _ = plt.hist(Idetvars[0][Idetvars[0].Run == 2][c], bins=b, density=True, histtype="step", linewidth=2, label="CV")
    for indd, d in enumerate(Idetvars[1:]):
        _ = plt.hist(d[d.Run == 2][c], bins=b, density=True, histtype="step", linewidth=2, label=IDETVAR_NAMES[indd+1])
    add_style(plt.gca(), l, det="ICARUS Run2", ylabel="Area Normalized")

    if DOSAVE:
        plt.savefig(PLOTDIR + "/png/ICARUS_detvar_%s_variations.png" % c, bbox_inches="tight")
        plt.savefig(PLOTDIR + "/pdf/ICARUS_detvar_%s_variations.pdf" % c, bbox_inches="tight")

In [ ]:
# ADD IN PID VARIATIONS
def v_variation(df, setvars):
    df = df[[c for c in df.columns if "univ" not in c]].copy()
    for (new, old) in setvars:
        df[new] = df[old]
    return df

def v_chi2smear(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2smear13_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2smear13_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2smear13_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2smear13_of_prot_cand"),
    ]
    return v_variation(df, setvars)


def v_chi2hi(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2hi_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2hi_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2hi_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2hi_of_prot_cand"),
    ]
    return v_variation(df, setvars)

In [ ]:
Schi2_detvars = [v_chi2smear(Sdf), v_chi2hi(Sdf)]
Ichi2_detvars = [v_chi2smear(Idf), v_chi2hi(Idf)]

SCHI2_DETVAR_NAMES = ["Smeared dE/dx", "Gain Hi"]
ICHI2_DETVAR_NAMES = ["Smeared dE/dx", "Gain Hi"]

In [ ]:
def simple_cosmic_rej(df):
    is_spine = "SPINE" in RECO
    return FV(df) & (df.crlongtrkdiry > -0.3)

def cosmic_cut(df):
    if "SPINE" not in RECO:
        return df.nu_score > 0.4
    else:
        return df.is_time_contained
        
def crtveto(df):
    return ~df.crthit

def twoprong_cut(df):
    return np.isnan(df.other_shw_length) & np.isnan(df.other_trk_length)

def pid_cut(df):
    is_spine = "SPINE" in RECO
    if not is_spine:
        return twoprong_cut(df) & gc.pid_cut_df(df)
    else:
        return twoprong_cut(df) & (df.prot_chi2_of_prot_cand > 0.6) & (df.mu_chi2_of_mu_cand > 0.6)

In [ ]:
def signalbox(df):
    return cosmic_cut(df) & twoprong_cut(df) & pid_cut(df) # & (~df.crthit)

In [ ]:
assert(np.all(np.isnan(Sdirt[~OOAVSBND(Sdirt)].true_vtx_x)))

In [ ]:
assert(len(Idirt) == 0 or np.all(np.isnan(Idirt[~OOAVICARUS(Idirt)].true_vtx_x)))

In [ ]:
Sdf["selected"] = signalbox(Sdf)
Sdirt["selected"] = signalbox(Sdirt)
Soffbeam["selected"] = signalbox(Soffbeam)

for i in range(len(Sdetvars)):
    Sdetvars[i]["selected"] = signalbox(Sdetvars[i])

for i in range(len(Schi2_detvars)):
    Schi2_detvars[i]["selected"] = signalbox(Schi2_detvars[i])

In [ ]:
Idf["selected"] = signalbox(Idf)
if len(Idirt) > 0:
    Idirt["selected"] = signalbox(Idirt)
else:
    Idirt["selected"] = False
Ioffbeam["selected"] = signalbox(Ioffbeam)

for i in range(len(Idetvars)):
    Idetvars[i]["selected"] = signalbox(Idetvars[i])

for i in range(len(Ichi2_detvars)):
    Ichi2_detvars[i]["selected"] = signalbox(Ichi2_detvars[i])

In [ ]:
Sdf["true_E"] = Sdf.true_E.fillna(-1)
Sdirt["true_E"] = Sdirt.true_E.fillna(-1)
Soffbeam["true_E"] = -1

for i in range(len(Sdetvars)):
    Sdetvars[i]["true_E"] = Sdetvars[i].true_E.fillna(-1)

for i in range(len(Schi2_detvars)):
    Schi2_detvars[i]["true_E"] = Schi2_detvars[i].true_E.fillna(-1)

In [ ]:
Idf["true_E"] = Idf.true_E.fillna(-1)
Idirt["true_E"] = Idirt.true_E.fillna(-1)
Ioffbeam["true_E"] = -1

for i in range(len(Idetvars)):
    Idetvars[i]["true_E"] = Idetvars[i].true_E.fillna(-1)

for i in range(len(Ichi2_detvars)):
    Ichi2_detvars[i]["true_E"] = Ichi2_detvars[i].true_E.fillna(-1)

In [ ]:
Ssystematics = [
    loaddf.FluxSystematic(Sdf),
    loaddf.XSecSystematic(Sdf),
    syst.NormalizationSystematic(0.02),
    syst.SystematicList(
        [syst.SampleSystematic(d, cvdf=Sdetvars[0]) for d in Sdetvars[1:]] + # detector variation samples
        [syst.SampleSystematic(d) for d in Schi2_detvars] # chi2 variations
    ),
    syst.SystSampleSystematic(Sdf[OOAVSBND(Sdf)]),
    syst.StatSampleSystematic(Soffbeam, norm=0.1) # TODO: change after unblinding. Simulate scaling up stats by 10x.
]

Isystematics = [
    loaddf.FluxSystematic(Idf),
    loaddf.XSecSystematic(Idf),
    syst.NormalizationSystematic(0.02),
    syst.SystematicList(
        [syst.SampleSystematic(d, cvdf=Idetvars[0]) for d in Idetvars[1:]] + # detector variation samples
        [syst.SampleSystematic(d) for d in Ichi2_detvars] # chi2 variations
    ),
    syst.SystSampleSystematic(Idf[OOAVICARUS(Idf)]),
    syst.StatSampleSystematic(Ioffbeam, norm=0.1) # TODO: change after unblinding. Simulate scaling up stats by 10x.
]

systematics = [
    syst.CorrelatedSystematic(loaddf.FluxSystematic(Sdf), loaddf.FluxSystematic(Idf)),
    syst.CorrelatedSystematic(loaddf.XSecSystematic(Sdf), loaddf.XSecSystematic(Idf)),

    syst.UnCorrelatedSystematic(
        syst.SystematicList(
                [syst.SampleSystematic(d, cvdf=Sdetvars[0]) for d in Sdetvars[1:]] + # detector variation samples
                [syst.SampleSystematic(d) for d in Schi2_detvars] + # chi2 variations
                [syst.SystSampleSystematic(Sdf[OOAVSBND(Sdf)]), syst.StatSampleSystematic(Soffbeam, norm=0.1)] # TODO: change after unblinding. Simulate scaling up stats by 10x.
        ),
        syst.SystematicList(
                [syst.SampleSystematic(d, cvdf=Idetvars[0]) for d in Idetvars[1:]] + # detector variation samples
                [syst.SampleSystematic(d) for d in Ichi2_detvars] + # chi2 variations
                [syst.SystSampleSystematic(Idf[OOAVICARUS(Idf)]), syst.StatSampleSystematic(Ioffbeam, norm=0.1)] # TODO: change after unblinding. Simulate scaling up stats by 10x.
        )),

    # POT norm is correlated SBND Run1 (1e20) - ICARUS Run4 (3e20), and uncorrelated ICARUS Run 2 (2e20)
    syst.SystematicList(
        [
            syst.UnCorrelatedSystematic(syst.NormalizationSystematic(0), syst.NormalizationSystematic(0.02*(2./5.))),
            syst.CorrelatedSystematic(syst.NormalizationSystematic(0.02), syst.NormalizationSystematic(0.02*(3./5.))),
        ]),
    
]


labels = [
    "Flux",
    "XSec",
    "POT Norm.",
    "Detector",
    "Dirt",
    "Beam Off",
    "Stat",
]

In [ ]:
# # Impact of CVwgt
# _ = plt.hist(Sdf.loc[Sdf[cut], "true_E"], weights=Sdf.loc[Sdf[cut], "glob_scale"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)
# _ = plt.hist(Sdf.loc[Sdf[cut], "true_E"], weights=Sdf.loc[Sdf[cut], "glob_scale"]/Sdf.loc[Sdf[cut], "cvwgt"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)

In [ ]:
# # Impact of CVwgt
# _ = plt.hist(Idf.loc[Idf[cut], "true_E"], weights=Idf.loc[Idf[cut], "glob_scale"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)
# _ = plt.hist(Idf.loc[Idf[cut], "true_E"], weights=Idf.loc[Idf[cut], "glob_scale"]/Idf.loc[Idf[cut], "cvwgt"], 
#              bins=np.linspace(0, 2, 11), histtype="step", linewidth=2)

In [ ]:
var = "nu_E_calo"
wgt = "glob_scale"
cut = "selected"
# bins = np.linspace(0, 1.5, 11)[2:]
bins = np.array([0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5])
bins = np.array([0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.25, 1.5])
centers = (bins[1:] + bins[:-1]) / 2

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut] & ~OOAVICARUS(Idf), var], bins=bins, weights=Idf.loc[Idf[cut] & ~OOAVICARUS(Idf), wgt], 
             histtype="step", linewidth=2, label="CV")
_ = plt.hist(Idf.loc[Idf[cut] & OOAVICARUS(Idf), var], bins=bins, weights=Idf.loc[Idf[cut] & OOAVICARUS(Idf), wgt], 
             histtype="step", linewidth=2, label="OOAV")
_ = plt.hist(Ioffbeam.loc[Ioffbeam[cut], var], bins=bins, weights=Ioffbeam.loc[Ioffbeam[cut], wgt], histtype="step", linewidth=2, label="Beam Off")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")
plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_altsamples.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_altsamples.pdf", bbox_inches="tight")
    

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut] & ~OOAVSBND(Sdf), var], bins=bins, weights=Sdf.loc[Sdf[cut] & ~OOAVSBND(Sdf), wgt], 
             histtype="step", linewidth=2, label="CV")
_ = plt.hist(Sdf.loc[Sdf[cut] & OOAVSBND(Sdf), var], bins=bins, weights=Sdf.loc[Sdf[cut] & OOAVSBND(Sdf), wgt], 
             histtype="step", linewidth=2, label="OOAV")
# _ = plt.hist(Sdirt.loc[Sdirt[cut], var], bins=bins, weights=Sdirt.loc[Sdirt[cut], wgt], histtype="step", linewidth=2, label="Dirt")
_ = plt.hist(Soffbeam.loc[Soffbeam[cut], var], bins=bins, weights=Soffbeam.loc[Soffbeam[cut], wgt], histtype="step", linewidth=2, label="Beam Off")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")
plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_altsamples.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_altsamples.pdf", bbox_inches="tight")
    

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(20):
    w = Sdf[wgt]*Sdf["flux_univ%i" % i]
    label = "Flux Univ's" if i == 0 else None
    _ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(20):
    w = Idf[wgt]*Idf["flux_univ%i" % i]
    label = "Flux Univ's" if i == 0 else None
    _ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]], histtype="step", linewidth=1, color="gray", label=label)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i,s in enumerate(loaddf.xsec_syst):
    w = Sdf[wgt]*Sdf["%s_univ" % s]
    label = "XSec Syst's" if i == 0 else None
    _ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
xsec_systs_toplot = loaddf.xsec_syst

In [ ]:
CV = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt])[0]


vs = []
ls = []

for i,s in enumerate(xsec_systs_toplot):
    w = Sdf[wgt]*Sdf["%s_univ" % s]
    v = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=w[Sdf[cut]])[0]
    _ = plt.hist(centers, bins=bins, weights=v/CV, histtype="step", linewidth=2)

plt.ylim([0.8, 1.2])
plt.axhline([1], color="gray", linestyle="--")


# add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
for s in xsec_systs_toplot:
    if np.any(np.isnan(Sdf["%s_univ" % s])):
        print(s, np.isnan(Sdf["%s_univ" % s]).sum())

In [ ]:
CV = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt])[0]

for i,s in enumerate(xsec_systs_toplot):
    w = Idf[wgt]*Idf["%s_univ" % s]
    v = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]])[0]
    _ = plt.hist(centers, bins=bins, weights=v/CV, histtype="step", linewidth=2)

plt.ylim([0.8, 1.2])
plt.axhline([1], color="gray", linestyle="--")


# add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i,s in enumerate(loaddf.xsec_syst):
    w = Idf[wgt]*Idf["%s_univ" % s]
    label = "XSec Syts's" if i == 0 else None
    _ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=w[Idf[cut]], histtype="step", linewidth=1, color="gray", label=label)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
def ratioplot(plt, detvars, names, title, presel=None, scales=None):
    fig, (ax_top, ax_bot) = plt.subplots(
      2, 1, sharex=True,
      gridspec_kw={"height_ratios": [3, 1], "hspace": 0},
    )
    
    centers = 0.5 * (bins[1:] + bins[:-1])
    
    cv_vals = None
    for i in range(len(detvars)):
        d = detvars[i]
        c = d[cut]
        if presel is not None:
            c = c & presel(d)
        
        n, _ = np.histogram(d.loc[c, var], bins=bins,
                          weights=d.loc[c, wgt])
        s = scales[i] if scales is not None else 1.
        n = n * s
        _ = ax_top.hist(centers, bins=bins, weights=n, color="tab:gray" if i == 0 else None,
                  histtype="step", linewidth=2, label=names[i])
        if i == 0:
            cv_vals = n
        else:
            ratio = np.divide(n, cv_vals, out=np.ones_like(n, dtype=float),
                            where=cv_vals > 0)
            _ = ax_bot.hist(centers, bins=bins, weights=ratio, 
                      histtype="step", linewidth=2)
    
    _ = ax_bot.axhline(1, color="gray", linestyle="--", linewidth=1)
    
    _ = add_style(ax_top, "", det=title)
    _ = ax_top.set_xlabel("")
    
    _ = ax_bot.set_xlabel("Reco. Neutrino Energy [GeV]", fontsize=FONTSIZE, fontweight="bold")
    _ = ax_bot.set_ylabel("Var. / Nom.", fontsize=FONTSIZE, fontweight="bold")
    _ = ax_bot.tick_params(axis="both", which="both", direction="in",
                     length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    _ = ax_bot.set_ylim([0.8, 1.2])
    for spine in ax_bot.spines.values():
        spine.set_linewidth(1.5)

In [ ]:
ratioplot(plt, Sdetvars, SDETVAR_NAMES, "SBND")

In [ ]:
ratioplot(plt, Idetvars, IDETVAR_NAMES, "ICARUS")

In [ ]:
ratioplot(plt, Idetvars, IDETVAR_NAMES, "ICARUS Run 2", lambda d: (d.Run == 2), scales=[Idetvars[0].shape[0]/I.shape[0] for I in Idetvars])

In [ ]:
# for i in range(len(Sdetvars)):
#     _ = plt.hist(Sdetvars[i].loc[Sdetvars[i][cut], var], bins=bins, weights=Sdetvars[i].loc[Sdetvars[i][cut], wgt], 
#             histtype="step", linewidth=2, label=SDETVAR_NAMES[i])

# add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt], histtype="step", linewidth=2, label="Nominal")

for i in range(len(Schi2_detvars)):
    _ = plt.hist(Schi2_detvars[i].loc[Schi2_detvars[i][cut], var], bins=bins, weights=Schi2_detvars[i].loc[Schi2_detvars[i][cut], wgt], 
            histtype="step", linewidth=2, label=SCHI2_DETVAR_NAMES[i])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND")

In [ ]:
_ = plt.hist(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt], histtype="step", linewidth=2, label="CV")

for i in range(len(Ichi2_detvars)):
    _ = plt.hist(Ichi2_detvars[i].loc[Ichi2_detvars[i][cut], var], bins=bins, weights=Ichi2_detvars[i].loc[Ichi2_detvars[i][cut], wgt], 
            histtype="step", linewidth=2, label=ICHI2_DETVAR_NAMES[i])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS")

In [ ]:
SCV = np.histogram(Sdf.loc[Sdf[cut], var], bins=bins, weights=Sdf.loc[Sdf[cut], wgt])[0]
Scovs = [s.cov(var, cut, bins, SCV) for s in Ssystematics]
Scovs.append(np.diag(SCV))

In [ ]:
ICV = np.histogram(Idf.loc[Idf[cut], var], bins=bins, weights=Idf.loc[Idf[cut], wgt])[0]
Icovs = [s.cov(var, cut, bins, ICV) for s in Isystematics]
Icovs.append(np.diag(ICV))

In [ ]:
S_detsyst_covs = [s.cov(var, cut, bins, SCV) for s in Ssystematics[labels.index("Detector")].systs]
I_detsyst_covs = [s.cov(var, cut, bins, ICV) for s in Isystematics[labels.index("Detector")].systs]

In [ ]:
for c, l in zip(Scovs, labels):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.5])

# plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
for c, l in zip(Icovs, labels):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.5])

# plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
for c, l in zip(Scovs, labels):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([1e-3, 8])

plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_log.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_log.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Scovs[0].shape)
for c, l in zip(S_detsyst_covs, SDETVAR_NAMES[1:] + SCHI2_DETVAR_NAMES):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)
    combined += c
    
for c, l in zip(Scovs[4:], labels[4:]):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/SCV), label=l, histtype="step", linewidth=2)
    combined += c

plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(combined))/SCV), label="Total", histtype="step", color="black", linewidth=2, linestyle="--")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2, legend_title="Uncorrelated Uncertainties")

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_uncorr.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Icovs[0].shape)
for c, l in zip(I_detsyst_covs, IDETVAR_NAMES[1:] + ICHI2_DETVAR_NAMES):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
    combined += c
    
for c, l in zip(Icovs[4:], labels[4:]):
    _ = plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(c))/ICV), label=l, histtype="step", linewidth=2)
    combined += c

plt.hist(centers, bins=bins, weights=(np.sqrt(np.diag(combined))/ICV), label="Total", histtype="step", color="black", linewidth=2, linestyle="--")

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2, legend_title="Uncorrelated Uncertainties")

plt.ylim([0, 0.125])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_uncorr.pdf", bbox_inches="tight")

In [ ]:
for c, l in zip(Icovs, labels):
    _ = plt.hist(centers[ICV>0], bins=bins, weights=(np.sqrt(np.diag(c))/ICV)[ICV>0], label=l, histtype="step", linewidth=2)


add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="ICARUS", ylabel="Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2)

plt.ylim([1e-3, 8])

plt.yscale("log")

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_log.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_log.pdf", bbox_inches="tight")

In [ ]:
covs = [s.cov(var, cut, bins, np.concatenate((SCV, ICV))) for s in systematics]
covs.append(np.diag(np.concatenate((SCV, ICV))))

In [ ]:
len(covs)

In [ ]:
def corr_f(cov):
    err = np.sqrt(np.diag(cov))
    err[err==0] = 1
    err_inv = np.diag(1/err)
    return err_inv@cov@err_inv

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib as mpl

In [ ]:
mpl.__version__

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[0])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)
        
        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("Flux Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/flux_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/flux_correlation.pdf")

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[1])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("XSec Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/xsec_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/xsec_correlation.pdf")

In [ ]:
plt.title("XSec Systematics")

plt.imshow(corr_f(covs[1]), origin="lower", vmin=0.8, vmax=1)
plt.colorbar()

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[2])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("Detector Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/detector_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/detector_correlation.pdf")

In [ ]:
fig, axs = plt.subplots(2, 2, layout='constrained',
    gridspec_kw={'wspace': 0.0, 'hspace': 0.0})
# plt.subplots_adjust(hspace=0, wspace=0)
fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0,
                            wspace=0)

C = corr_f(covs[3])

nbin = len(bins) - 1
for i, ax in enumerate(axs):
    for j, a in enumerate(ax):
        ic = 1 if i == 0 else 0
        cov = C[nbin*ic:nbin*(ic+1), nbin*j:nbin*(j+1)]
        pcol = a.pcolormesh(bins, bins, cov, cmap="bwr", vmin=-1, vmax=1, linewidth=0, rasterized=True)

        a.tick_params(axis='both', which='major', labelsize=FONTSIZE-2)
        if i == 0 and j == 0:
            _ = a.set_xticks([])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_ylabel("ICARUS", size=FONTSIZE-2)
        if i == 0 and j == 1:
            _ = a.set_xticks([])
            _ = a.set_yticks([])
        if i == 1 and j == 0:
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_yticks(np.linspace(bins[0], bins[-1], 7)[:-1])
            _ = a.set_ylabel("SBND", size=FONTSIZE-2)
            _ = a.set_xlabel("SBND", size=FONTSIZE-2)
        elif i == 1 and j == 1:
            pass
            _ = a.set_xticks(np.linspace(bins[0], bins[-1], 7)[1:])
            _ = a.set_yticks([])
            _ = a.set_xlabel("ICARUS", size=FONTSIZE-2)


_ = fig.suptitle("POT Norm. Systematic Correlation", size=FONTSIZE+2) #, x=0.46, y=0.95)
_ = fig.supxlabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)
_ = fig.supylabel("Reco. Neutrino Energy [GeV]", size=FONTSIZE)

norm = mcolors.Normalize(vmin=-1, vmax=1)
sm = cm.ScalarMappable(cmap="bwr", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=[a for ax in axs for a in ax])
cbar.set_label("Correlation", rotation=270, size=FONTSIZE-2, labelpad=12)

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/norm_correlation.png")
    plt.savefig(PLOTDIR + "/pdf/norm_correlation.pdf")

In [ ]:
def ratio_cov_full(x, y, cov):
    """
    Covariance of r = x / y given the full covariance of (x, y).

    Parameters
    ----------
    x, y : array-like, shape (n,)
        Central values
    cov : array-like, shape (2n, 2n)
        Full covariance matrix of (x, y)

        Ordering must be:
        cov = [[Cov(x,x), Cov(x,y)],
               [Cov(y,x), Cov(y,y)]]

    Returns
    -------
    cov_r : ndarray, shape (n, n)
        Covariance matrix of r
    """
    n = len(x)
    assert cov.shape == (2*n, 2*n)

    # Protect against division by zero
    eps = 1e-12
    y_safe = np.where(np.abs(y) < eps, eps, y)

    Dx = np.diag(1.0 / y_safe)
    Dy = np.diag(-x / y_safe**2)

    # Full Jacobian: shape (n, 2n)
    J = np.hstack([Dx, Dy])

    return J @ cov @ J.T

In [ ]:
ratio = SCV / ICV
ratio = ratio / ratio.mean()

ratio_cov_flux = ratio_cov_full(SCV, ICV, covs[0])
ratio_cov_xsec = ratio_cov_full(SCV, ICV, covs[1])
ratio_cov_det = ratio_cov_full(SCV, ICV, covs[2])
ratio_cov_norm = ratio_cov_full(SCV, ICV, covs[3])
ratio_cov_stat = ratio_cov_full(SCV, ICV, covs[4])
ratio_cov_all = ratio_cov_full(SCV, ICV, np.sum(covs, axis=0))

In [ ]:
_ = plt.hist(centers, bins=bins, weights=ratio, histtype="step", linewidth=2)
add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND / ICARUS", ylabel="SBND/ICARUS Ratio")
plt.axhline([1], color="black", linestyle="--")

In [ ]:
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_flux))/ratio, histtype="step", linewidth=2, label="Flux")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_xsec))/ratio, histtype="step", linewidth=2, label="XSec")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_det))/ratio, histtype="step", linewidth=2, label="Detector")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_norm))/ratio, histtype="step", linewidth=2, label="POT Norm.")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_stat))/ratio, histtype="step", linewidth=2, label="Stat.")
_ = plt.hist(centers, bins=bins, weights=np.sqrt(np.diag(ratio_cov_all))/ratio, histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

add_style(plt.gca(), "Reco. Neutrino Energy [GeV]", det="SBND / ICARUS", 
          ylabel="Uncertainty on SBND/ICARUS Ratio", legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.15])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ratio_signalbox_systematics.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ratio_signalbox_systematics.pdf", bbox_inches="tight")

In [ ]:
reco_bins = bins
true_bins = np.array([-2] + list(reco_bins) + [np.inf])
true_bins_centers = (true_bins[:-1] + true_bins[1:])/2
true_bins_centers, true_bins

In [ ]:
SCV2 = np.histogramdd([Sdf.loc[Sdf[cut], "nu_E_calo"], Sdf.loc[Sdf[cut], "true_E"]], 
                      bins=[reco_bins, true_bins], weights=Sdf.loc[Sdf[cut], wgt])[0]

ICV2 = np.histogramdd([Idf.loc[Idf[cut], "nu_E_calo"], Idf.loc[Idf[cut], "true_E"]], 
                      bins=[reco_bins, true_bins], weights=Idf.loc[Idf[cut], wgt])[0]

SCVT = np.histogram(Sdf.loc[Sdf[cut], "true_E"], bins=true_bins, weights=Sdf.loc[Sdf[cut], wgt])[0]
ICVT = np.histogram(Idf.loc[Idf[cut], "true_E"], bins=true_bins, weights=Idf.loc[Idf[cut], wgt])[0]

# CV2D = np.concatenate((SCV2.T, ICV2.T), axis=0) # .flatten()
# CV2D = np.concatenate((CV2D, CV2D), axis=1)

Imig = ICV2.T / ICV2.sum(axis=1)
Smig = SCV2.T / SCV2.sum(axis=1)

migration = np.block([
    [Smig, np.zeros(Smig.shape)],
    [np.zeros(Imig.shape), Imig]
])

def unfold(C):
    return migration@C@migration.T

def unfoldSBND(C):
    return Smig@C@Smig.T

def unfoldICARUS(C):
    return Imig@C@Imig.T

In [ ]:
plt.imshow(migration, origin="lower")

In [ ]:
ratioT = SCVT / ICVT
ratioT = ratioT / ratioT.mean()

ratioT_cov_flux = ratio_cov_full(SCVT, ICVT, unfold(covs[0]))
ratioT_cov_xsec = ratio_cov_full(SCVT, ICVT, unfold(covs[1]))
ratioT_cov_det = ratio_cov_full(SCVT, ICVT, unfold(covs[2]))
ratioT_cov_norm = ratio_cov_full(SCVT, ICVT, unfold(covs[3]))
ratioT_cov_stat = ratio_cov_full(SCVT, ICVT, unfold(covs[4]))
ratioT_cov_all = ratio_cov_full(SCVT, ICVT, unfold(np.sum(covs, axis=0)))

In [ ]:
plot_bins = true_bins[1:-1]

_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_flux))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="Flux")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_xsec))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="XSec")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_det))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="Detector")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_norm))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="POT Norm.")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_stat))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="Stat.")
_ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(ratioT_cov_all))/ratioT)[1:-1],
             histtype="step", linewidth=2, label="All")
# plt.ylim([0, 0.25])

add_style(plt.gca(), "True Neutrino Energy [GeV]", det="SBND / ICARUS", 
          ylabel="Unfolded Unc. on SBND/ICARUS Ratio", legend_loc="upper center", legend_ncol=2)

plt.ylim([0, 0.15])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ratio_signalbox_systematics_trueE.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ratio_signalbox_systematics_trueE.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Icovs[0].shape)
for c, l in zip(I_detsyst_covs, IDETVAR_NAMES[1:] + ICHI2_DETVAR_NAMES):
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(c)))/ICVT)[1:-1], label=l, histtype="step", linewidth=2)
    combined += c
    
for c, l in zip(Icovs[4:], labels[4:]):
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(c)))/ICVT)[1:-1], label=l, histtype="step", linewidth=2)
    combined += c

plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldICARUS(combined)))/ICVT)[1:-1], label="Total", histtype="step", color="black", linewidth=2, linestyle="--")

add_style(plt.gca(), "True Neutrino Energy [GeV]", det="ICARUS", ylabel="Unfolded Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2, legend_title="Uncorrelated Uncertainties")

plt.ylim([0, 0.08])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/ICARUS_signalbox_systematics_trueE_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/ICARUS_signalbox_systematics_trueE_uncorr.pdf", bbox_inches="tight")

In [ ]:
combined = np.zeros(Scovs[0].shape)
for c, l in zip(S_detsyst_covs, SDETVAR_NAMES[1:] + SCHI2_DETVAR_NAMES):
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldSBND(c)))/SCVT)[1:-1], label=l, histtype="step", linewidth=2)
    combined += c
    
for c, l in zip(Scovs[4:], labels[4:]):
    _ = plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldSBND(c)))/SCVT)[1:-1], label=l, histtype="step", linewidth=2)
    combined += c

plt.hist(true_bins_centers[1:-1], bins=plot_bins, weights=(np.sqrt(np.diag(unfoldSBND(combined)))/SCVT)[1:-1], label="Total", histtype="step", color="black", linewidth=2, linestyle="--")

add_style(plt.gca(), "True Neutrino Energy [GeV]", det="SBND", ylabel="Unfolded Fractional Uncertainty", 
          legend_loc="upper center", legend_ncol=2, legend_title="Uncorrelated Uncertainties")

plt.ylim([0, 0.08])

if DOSAVE:
    plt.savefig(PLOTDIR + "/png/SBND_signalbox_systematics_trueE_uncorr.png", bbox_inches="tight")
    plt.savefig(PLOTDIR + "/pdf/SBND_signalbox_systematics_trueE_uncorr.pdf", bbox_inches="tight")